This is following - 113  

Part A: We will divide our documents into CHUNKS

In [1]:
import os
import glob 
import tiktoken
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from sklearn.manifold import TSNE
import plotly.graph_objects as go
 

/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Langchain related Imports 

from langchain_chroma import Chroma # the newer, recommended approach
# from langchain_community.vectorstores import Chroma - older approach


from langchain_openai import OpenAIEmbeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_18615/3474719954.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [3]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


Let's read in all  data into a dictionary

In [4]:
# How many files in all the documents?

knowledge_base_path = "knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

Found 15 files in the knowledge base


In [5]:
# How many characters in all the documents?

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Total characters in knowledge base: 18,195


In [6]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode( entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 3,400


We could think like why cant we pass these whole token as single context to gpt models as the latest gpt models can handle these many tokens easily . 

But we need to understand if the document is very huge , then sending the whole context for a simple question which is present across few tokens is waste of resources . 

In [23]:
# Load in everything in the knowledgebase using LangChain's loaders

knowledge_base_path = "knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)

#folders = glob.glob("knowledge_base/*")
documents = []
for file in files:
    doc_type = os.path.basename(file)
    #print(f"folder :{file} , doc_cat : {doc_type}")
    #loader = DirectoryLoader(file, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    loader = TextLoader(file)
    loader_docs = loader.load()
    for doc in loader_docs:
        #doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")
 

Loaded 15 documents


In [24]:
documents[0]

Document(metadata={'source': 'knowledge_base/Shari_Professional.md'}, page_content="# Shari Thomas – Professional and Academic Background\n\nShari Thomas is a technology professional with experience in higher education, software development, data science, and artificial intelligence. Her career reflects a combination of academic teaching, practical technical experience, and a continuing interest in developing skills relevant to the modern AI industry.\n\nShari completed her Bachelor of Technology (B.Tech) degree in Computer Science and Engineering. She subsequently obtained a Master of Technology (M.Tech) degree with a specialization in Image Processing. Her educational background introduced her to a range of computing disciplines, including software engineering, computer science fundamentals, digital image processing, and technologies associated with artificial intelligence.\n\nFollowing her postgraduate education, Shari began her professional career in the field of teaching. She work

In [25]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 92 chunks
First chunk:

page_content='# Shari Thomas – Professional and Academic Background' metadata={'source': 'knowledge_base/Shari_Professional.md'}


PART B: Make vectors and store in Chroma

Huggingface embedding is one of the most popular one.

In [26]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:01<00:00, 70.24it/s]


Vectorstore created with 92 documents
